In [5]:
# =========================
# 0) SETUP UNIVERSAL — Drive o descarga ZIP (accesible sin tu Drive)
# =========================

# ---------- Configuración editable ----------
ZIP_URL = "https://github.com/KIRZONMAN/Parcial_Modelos_MBA/archive/refs/heads/main.zip"  # enlace público al zip
NOMBRE_CARPETA_PROYECTO = "Parcial1_MBA"  # nombre de la carpeta del proyecto
RUTAS_DRIVE_CANDIDATAS = [  # rutas donde buscar el proyecto en drive
    "/content/drive/MyDrive/Colab_Notebooks/Parcial1_MBA",
    "/content/drive/MyDrive/Colab Notebooks/Parcial1_MBA",
]
# --------------------------------------------

import os  # rutas y listdir
import sys  # sys.path
import shutil  # move, rmtree

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)  # montar drive en colab
except Exception:
    pass  # fuera de colab no hay drive

RUTA_PROYECTO = None
for candidata in RUTAS_DRIVE_CANDIDATAS:
    if os.path.exists(candidata):  # si existe usamos esa
        RUTA_PROYECTO = candidata
        print("✅ Usando proyecto desde Drive:", RUTA_PROYECTO)
        break

if RUTA_PROYECTO is None:
    RUTA_PROYECTO = "/content/" + NOMBRE_CARPETA_PROYECTO  # vamos a descargar ahí

    if not os.path.exists(RUTA_PROYECTO):  # no está en drive ni en content
        if not ZIP_URL or not ZIP_URL.strip():
            # error si no hay zip configurado ni proyecto en /content
            raise RuntimeError(
                "❌ No hay proyecto en Drive ni en /content. "
                "Pega en ZIP_URL el enlace público al ZIP del proyecto "
                "(ej. GitHub: .../archive/refs/heads/main.zip)."
            )

        import urllib.request
        import zipfile

        zip_path = "/content/proyecto.zip"
        print("⬇️ Descargando proyecto desde ZIP...")
        urllib.request.urlretrieve(ZIP_URL.strip(), zip_path)  # descarga a disco

        pre_dirs = set(os.listdir("/content"))  # estado antes de extraer

        print("📦 Descomprimiendo en /content/ ...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall("/content/")  # extrae todo en content

        os.remove(zip_path)  # borrar zip ya extraído

        post_dirs = set(os.listdir("/content"))
        nuevas = [  # carpetas nuevas creadas por el zip
            d for d in (post_dirs - pre_dirs)
            if os.path.isdir(os.path.join("/content", d))
        ]

        candidatas = []
        for d in nuevas:
            if d.startswith("."):
                continue  # ignorar ocultas
            full = os.path.join("/content", d)
            if os.path.isdir(os.path.join(full, "motor")) or os.path.exists(os.path.join(full, "Entregable_Parcial_MBA.ipynb")):
                candidatas.append(d)  # parece nuestro proyecto

        if not candidatas:  # fallback: buscar en /content a mano
            for d in os.listdir("/content"):
                if d.startswith(".") or d in ("drive", "sample_data"):
                    continue  # saltar drive y sample_data
                full = os.path.join("/content", d)
                if os.path.isdir(full) and (os.path.isdir(os.path.join(full, "motor")) or os.path.exists(os.path.join(full, "Entregable_Parcial_MBA.ipynb"))):
                    candidatas.append(d)

        if not candidatas:
            # no se encontró carpeta del proyecto tras descomprimir
            raise FileNotFoundError(
                "❌ No pude identificar la carpeta del proyecto tras descomprimir. "
                "Revisa el ZIP_URL o el contenido del ZIP."
            )

        src = os.path.join("/content", candidatas[0])  # primera candidata

        if os.path.exists(RUTA_PROYECTO):
            shutil.rmtree(RUTA_PROYECTO)  # limpiar si ya existía

        shutil.move(src, RUTA_PROYECTO)
        print("✅ Descargado y ubicado desde ZIP en:", RUTA_PROYECTO)

os.chdir(RUTA_PROYECTO)  # trabajar en la carpeta del proyecto
if RUTA_PROYECTO not in sys.path:
    sys.path.insert(0, RUTA_PROYECTO)  # para importar motor, universidad, trafico

for carpeta in ["motor", "universidad", "trafico", "figuras"]:
    os.makedirs(carpeta, exist_ok=True)  # crear si no existen

for p in ["motor/__init__.py", "universidad/__init__.py", "trafico/__init__.py"]:
    with open(p, "w", encoding="utf-8") as f:
        f.write("# package init\n")  # init vacío

def _verificar_py_no_json(ruta: str) -> None:  # que el .py no sea en realidad un notebook
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"❌ No existe {ruta}. Revisa ruta o ZIP.")
    with open(ruta, "r", encoding="utf-8") as f:
        cabeza = f.read(400).lstrip()  # primeras líneas
    if cabeza.startswith("{") and '"nbformat"' in cabeza[:200]:
        raise AssertionError(f"❌ {ruta} parece JSON de notebook; debe ser .py.")  # es un .ipynb renombrado

for _r in [  # comprobar que los .py son código y no json
    "trafico/luces_trafico.py",
    "trafico/agentes_trafico.py",
    "trafico/sim_trafico.py",
    "trafico/metricas_trafico.py",
]:
    _verificar_py_no_json(_r)  # lanza si falta o es json

print("✅ Contenido raíz:", os.listdir("."))
print("✅ Módulos verificados.")  # fin setup

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Usando proyecto desde Drive: /content/drive/MyDrive/Colab_Notebooks/Parcial1_MBA
✅ Contenido raíz: ['motor', 'universidad', 'trafico', 'figuras', 'Copia de Entregable_Parcial_MBA.ipynb', 'Entregable_Parcial_MBA.ipynb']
✅ Módulos verificados.


## Imports y descripción de modelos

**Qué hace el setup (celda anterior):** Elige la ruta del proyecto (Drive o descarga ZIP), añade la ruta a `sys.path`, crea carpetas y `__init__.py`, y verifica que los módulos de tráfico sean .py y no JSON.

**Universidad:** Grid con zonas (Aula, Biblioteca, Cafetería). Agentes = estudiantes; 1 por celda. Reglas: umbral de hambre, afinidad por zona, permanencia, capacidad por zona. Métricas: ocupación por zona por tick, % tiempo por zona por agente.

**Tráfico:** Grid con fondo VACÍO, cruz de vías e intersección central. Agentes = vehículos; 1 por celda. Semáforos por fases (verde/amarillo/rojo). Spawn en bordes sobre VÍA; accidentes por violación (amarillo/rojo) y ocupación. Métricas: cola y flujo por tick, throughput, estado semáforo.

**Restricciones:** Tick discreto; 1 agente por celda; invariantes y validaciones; reproducible con semilla; eventos con límite (selector UI, sin spam).

**Supuestos y limitaciones:** (1) Movimientos son teleport entre celdas de la misma zona (Universidad) o avance celda a celda (Tráfico). (2) Semáforo: solo una fase en verde; amarillo/rojo con probabilidad de violación. (3) Accidentes solo por violación + celda ocupada. (4) Sin dependencias pesadas; solo numpy, matplotlib, ipywidgets.

In [6]:
# =========================
# 1) Imports
# =========================

from motor.grid import MundoCuadricula, TipoCelda  # grid y tipos de celda
from motor.sim_base import SimulacionBase, AgenteBase  # base de simulación
from motor.viz import dibujar_cuadricula, dibujar_agentes  # dibujo del grid
from universidad.sim_uni import SimulacionUniversidad  # sim de universidad
import importlib

import universidad.metricas_uni as _mu
importlib.reload(_mu)  # por si se tocó el módulo

resumen_porcentaje_tiempo_por_zona = _mu.resumen_porcentaje_tiempo_por_zona
graficar_ocupacion = _mu.graficar_ocupacion  # gráfica ocupación por zona
primeros_eventos = getattr(_mu, "primeros_eventos", lambda eventos, n=20: list(eventos[:n]))  # fallback si no existe
ultimos_eventos = getattr(_mu, "ultimos_eventos", lambda eventos, n=20: list(eventos[-n:]) if len(eventos) >= n else list(eventos))  # fallback

def _filtrar_fallback(eventos, id_estudiante):  # filtro simple por texto
    marca = f"Estudiante {id_estudiante} "
    return [e for e in eventos if marca in e]

filtrar_eventos_por_estudiante = getattr(_mu, "filtrar_eventos_por_estudiante", _filtrar_fallback)
import trafico.agentes_trafico as _at
importlib.reload(_at)
import trafico.sim_trafico as _st
importlib.reload(_st)
SimulacionTrafico = _st.SimulacionTrafico  # clase de sim tráfico
crear_mapa_trafico = _st.crear_mapa_trafico  # crea grid con vías
import inspect
sig = inspect.signature(_at.Vehiculo)  # para comprobar si tiene intencion giro
if "intencion_giro" not in sig.parameters and "intencion" not in sig.parameters:
    print("⚠️ Vehiculo no tiene campos de giro; revisa trafico/agentes_trafico.py")
import trafico.metricas_trafico as _mt
importlib.reload(_mt)
tiempo_espera_promedio = _mt.tiempo_espera_promedio
flujo_total = _mt.flujo_total
graficar_cola_y_flujo = _mt.graficar_cola_y_flujo  # cola y flujo vs tick
graficar_movimientos = getattr(_mt, "graficar_movimientos", lambda lst: None)
graficar_mapa_con_accidente = getattr(_mt, "graficar_mapa_con_accidente", lambda s: None)
graficar_estado_trafico = getattr(_mt, "graficar_estado_trafico", graficar_mapa_con_accidente)
resumen_accidentes = getattr(_mt, "resumen_accidentes", lambda s: {})
print("✅ Imports listos.")  # fin imports

✅ Imports listos.


## Parámetros globales
Semillas, pasos, probabilidades de tráfico y opciones de animación. El panel usa estos valores por defecto.

In [7]:
# Parámetros globales: semillas (reproducibilidad), pasos, probabilidades de tráfico y animación
SEMILLA_UNI = 777  # semilla numpy para universidad
SEMILLA_TRAFICO = 449  # semilla para tráfico
PASOS_UNI = 200  # cuántos ticks corre la sim universidad
MAX_TICKS = 150  # ticks máximos tráfico
PASOS_TRAFICO = MAX_TICKS  # mismo valor
P_PASARSE_AMARILLO = 0.03  # prob. pasarse amarillo (por perfil riesgo)
P_PASARSE_ROJO = 0.003  # prob. pasarse rojo
ANIMAR = True  # si False Run all más rápido; panel puede animar con checkbox
SNAPSHOTS_CADA_N = 10  # cada cuántos ticks un frame si se anima

## Panel de control (ejecución principal)

**Qué hace el panel:** Al pulsar Ejecutar se recrean entornos y simulaciones desde cero; se ejecuta la simulación según el modo (Universidad / Tráfico / Ambos). Según los checkboxes: **Mostrar resultados** dibuja las gráficas de métricas; **Animar** muestra snapshots cada N ticks; **Mostrar eventos** abre el selector para ver primeros/últimos N o por agente.

**Qué muestran las métricas:** Universidad: % de tiempo por zona por estudiante y ocupación por zona vs tick. Tráfico: cola y flujo por tick, throughput (vehículos salidos), tiempo de espera; si hubo accidente, tipo de violación (amarillo/rojo).

Como recomendación, para ver una mayor fluidez dentro de la simulación, recomiendo bajar velocidad y ticks / frame al minimo

In [8]:
# =========================
# Panel de control — ejecutar sin editar celdas (recrea todo desde cero)
# =========================
try:  # intentar cargar panel con ipywidgets
    import re  # regex para extraer id veh en eventos
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    import matplotlib.pyplot as plt
    output_resultados = widgets.Output()  # zona donde se pintan resultados/gráficas
    output_eventos = widgets.Output()  # zona del selector de eventos
    ultimo_sim_uni, ultimo_sim_traf = None, None  # para mostrar eventos después
    def construir_selector_eventos(sim, titulo="Eventos", maxN=100, tipo="uni"):
        ev = getattr(sim, "eventos", [])  # lista de eventos de la sim
        out_ev = widgets.Output()  # donde se imprimen los eventos
        modo = widgets.Dropdown(options=[("Primeros N", "primeros"), ("Últimos N", "ultimos"), ("Por agente", "agente")], value="primeros", description="Ver:")
        n_slider = widgets.IntSlider(value=min(20, maxN), min=1, max=maxN, description="N:")
        if tipo == "uni" and hasattr(sim, "agentes"):
            ids = [(str(i), i) for i in range(len(sim.agentes))]  # ids estudiantes
        else:
            ids = []
            for e in ev:
                s = str(e)
                m = re.search(r"\bVeh\s+(\d+)\b", s)  # extraer id vehículo del evento
                if m:
                    vid = int(m.group(1))
                    if not any(x[1] == vid for x in ids):
                        ids.append((str(vid), vid))
            ids = sorted(ids, key=lambda x: x[1]) if ids else [(str(0), 0)]  # fallback un id
        id_agente = widgets.Dropdown(options=ids, description="ID agente:" if tipo == "uni" else "ID veh:", disabled=True)
        def on_modo(change):
            id_agente.disabled = (change.new != "agente")  # habilitar solo en modo agente
        modo.observe(on_modo, names="value")  # habilitar id_agente solo en modo agente
        def filtrar_traf(evs, vid):
            return [e for e in evs if re.search(fr"\bVeh\s+{vid}\b", str(e))]  # eventos de un veh
        def _mostrar(b):
            with out_ev:
                clear_output(wait=True)
                if modo.value == "primeros":
                    for e in primeros_eventos(ev, n=n_slider.value):
                        print(e)
                elif modo.value == "ultimos":
                    for e in ultimos_eventos(ev, n=n_slider.value):
                        print(e)
                else:
                    if tipo == "uni":
                        for e in filtrar_eventos_por_estudiante(ev, id_agente.value):
                            print(e)
                    else:
                        for e in filtrar_traf(ev, id_agente.value):
                            print(e)
        btn = widgets.Button(description="Mostrar")
        btn.on_click(_mostrar)  # al pulsar imprime en out_ev
        return widgets.VBox([widgets.HTML("<b>" + titulo + "</b>"), widgets.HBox([modo, n_slider, id_agente]), btn, out_ev])  # layout del selector
    _modo = widgets.Dropdown(options=[("Universidad", "uni"), ("Tráfico", "trafico"), ("Ambos", "ambos")], value="ambos", description="Modo:")
    _s_uni = widgets.IntText(value=SEMILLA_UNI, description="Semilla Uni:")
    _s_traf = widgets.IntText(value=SEMILLA_TRAFICO, description="Semilla Tráf:")
    _pasos_uni = widgets.IntText(value=PASOS_UNI, description="Pasos Uni:")
    _max_ticks = widgets.IntText(value=MAX_TICKS, description="Max ticks Tráf:")
    _p_amarillo = widgets.FloatText(value=P_PASARSE_AMARILLO, description="p_amarillo:")
    _p_rojo = widgets.FloatText(value=P_PASARSE_ROJO, description="p_rojo:")
    _p_spawn = widgets.FloatText(value=0.25, description="p_spawn:")
    _max_veh = widgets.IntText(value=40, description="max_veh:")
    _cola_umbral = widgets.IntText(value=12, description="cola_umbral:")
    _factor_red = widgets.FloatText(value=0.5, description="factor_red:")
    _p_spawn_min = widgets.FloatText(value=0.05, description="p_spawn_min:")
    _p_giro_izq = widgets.FloatText(value=0.15, description="p_giro_izq:")
    _p_giro_der = widgets.FloatText(value=0.20, description="p_giro_der:")
    _velocidad_anim = widgets.FloatSlider(min=0.02, max=0.40, step=0.01, value=0.12, description="velocidad:")
    _ticks_por_frame = widgets.IntSlider(min=1, max=10, step=1, value=2, description="ticks/frame:")
    _mostrar_resultados = widgets.Checkbox(value=True, description="Mostrar resultados")
    _animar = widgets.Checkbox(value=False, description="Animar")
    _mostrar_eventos = widgets.Checkbox(value=False, description="Mostrar eventos")
    def _ejecutar(b):  # callback del botón Ejecutar
        # Recrear todo desde cero según modo elegido
        global entorno_uni, sim_uni, entorno_trafico, spawns_trafico, sim_trafico, ultimo_sim_uni, ultimo_sim_traf
        modo = _modo.value
        with output_resultados:
            clear_output(wait=True)
            # --- Universidad: mapa 30x20 (aula, biblio, cafetería), sim y opcional animación ---
            if modo in ("uni", "ambos"):
                entorno_uni = MundoCuadricula(ancho=30, alto=20)
                entorno_uni.rellenar_rectangulo(2, 2, 16, 10, TipoCelda.AULA)  # aula
                entorno_uni.rellenar_rectangulo(20, 2, 8, 8, TipoCelda.BIBLIOTECA)  # biblio
                entorno_uni.rellenar_rectangulo(20, 12, 8, 6, TipoCelda.CAFETERIA)  # café
                # crea sim universidad con caps y probs del panel
                sim_uni = SimulacionUniversidad(entorno=entorno_uni, n_estudiantes=25, semilla=_s_uni.value,
                    cap_aula=20, cap_biblio=15, cap_cafe=10, umbral_aula=18, p_volver_aula=0.10,
                    enfriamiento_mov=2, p_ir_cafeteria=0.30)
                if _animar.value:
                    pasos_uni_anim = int(_pasos_uni.value)
                    for _ in range(pasos_uni_anim):
                        sim_uni.paso()
                        sim_uni.tiempo += 1
                        if sim_uni.tiempo % SNAPSHOTS_CADA_N == 0:  # cada N ticks un frame
                            clear_output(wait=True)
                            fig, ax = plt.subplots(figsize=(7, 5))
                            dibujar_cuadricula(ax, sim_uni.entorno.terreno)
                            posiciones = [sim_uni.entorno.obtener_posicion_agente(aid) for aid in sim_uni.agentes]
                            dibujar_agentes(ax, posiciones)
                            ax.set_title("Universidad t=" + str(sim_uni.tiempo))
                            plt.show()
                            plt.pause(0.05)
                else:
                    sim_uni.ejecutar(pasos=_pasos_uni.value, verificar_cada=10)
                ultimo_sim_uni = sim_uni  # para selector de eventos
                if _mostrar_resultados.value:
                    print("✅ Universidad terminada. Tiempo por zona (%):", resumen_porcentaje_tiempo_por_zona(sim_uni.agentes))
                    graficar_ocupacion(sim_uni.metricas)  # ocupación por zona vs tick
                    plt.show()
            # --- Tráfico: mapa 2x2, spawns en bordes, sim y opcional animación ---
            if modo in ("trafico", "ambos"):
                entorno_trafico, spawns_trafico = crear_mapa_trafico(ancho=25, alto=25, ancho_via=3, tam_interseccion=5, layout="2x2")  # mapa y puntos spawn
                # sim tráfico con probs y spawns del panel
                sim_trafico = SimulacionTrafico(entorno=entorno_trafico, semilla=_s_traf.value, max_vehiculos=_max_veh.value,
                    p_pasarse_amarillo=_p_amarillo.value, p_pasarse_rojo=_p_rojo.value, p_spawn=_p_spawn.value,
                    cola_umbral=_cola_umbral.value, factor_reduccion_spawn=_factor_red.value, p_spawn_min=_p_spawn_min.value,
                    p_giro_izq=_p_giro_izq.value, p_giro_der=_p_giro_der.value,
                    spawns_por_direccion=spawns_trafico)
                if _animar.value:
                    max_ticks_anim = int(_max_ticks.value)
                    steps_hechos = 0
                    delay = _velocidad_anim.value
                    tpf = max(1, _ticks_por_frame.value)  # ticks por frame
                    while steps_hechos < max_ticks_anim:
                        for _ in range(tpf):
                            if sim_trafico.detener:
                                break
                            sim_trafico.paso()
                            sim_trafico.tiempo += 1
                            steps_hechos += 1
                            if steps_hechos >= max_ticks_anim:
                                break
                        clear_output(wait=True)
                        graficar_estado_trafico(sim_trafico)  # mapa con vehículos y semáforos
                        plt.show()
                        if sim_trafico.hubo_accidente:
                            plt.pause(2)
                            break
                        plt.pause(delay)
                else:
                    sim_trafico.ejecutar(pasos=_max_ticks.value, verificar_cada=10)
                ultimo_sim_traf = sim_trafico  # para selector eventos
                if _mostrar_resultados.value:
                    print("✅ Tráfico terminado. Salidos:", sim_trafico.vehiculos_salidos)
                    graficar_cola_y_flujo(sim_trafico.cola_por_tick, sim_trafico.flujo_por_tick)
                    plt.show()
                    graficar_estado_trafico(sim_trafico)  # estado final
                    plt.show()
                    if sim_trafico.hubo_accidente:
                        print("⚠️ Accidente tick", sim_trafico.tick_accidente, "|", sim_trafico.info_accidente.get("tipo_violacion", "?"))
        with output_eventos:
            clear_output(wait=True)
            if _mostrar_eventos.value:
                if modo == "uni" and ultimo_sim_uni is not None:
                    display(construir_selector_eventos(ultimo_sim_uni, titulo="Eventos Universidad", maxN=100, tipo="uni"))
                elif modo == "trafico" and ultimo_sim_traf is not None:
                    display(construir_selector_eventos(ultimo_sim_traf, titulo="Eventos Tráfico", maxN=100, tipo="traf"))
                elif modo == "ambos":
                    if ultimo_sim_uni is not None:
                        display(construir_selector_eventos(ultimo_sim_uni, titulo="Eventos Universidad", maxN=100, tipo="uni"))
                    if ultimo_sim_traf is not None:
                        display(construir_selector_eventos(ultimo_sim_traf, titulo="Eventos Tráfico", maxN=100, tipo="traf"))
    _btn = widgets.Button(description="Ejecutar")
    _btn.on_click(_ejecutar)  # al pulsar corre la sim
    display(widgets.VBox([widgets.HBox([_modo, _s_uni, _s_traf, _pasos_uni, _max_ticks]),
                          widgets.HBox([_p_amarillo, _p_rojo, _p_spawn, _max_veh]),
                          widgets.HBox([_cola_umbral, _factor_red, _p_spawn_min, _p_giro_izq, _p_giro_der]),
                          widgets.HBox([_velocidad_anim, _ticks_por_frame]),
                          widgets.HBox([_mostrar_resultados, _animar, _mostrar_eventos, _btn]),
                          widgets.HTML("<b>Resultados / gráficas</b>"), output_resultados,
                          widgets.HTML("<b>Selector de eventos</b>"), output_eventos]))  # layout completo del panel
except Exception as e:
    print("Panel de control (ipywidgets):", e)
    try:
        if _mostrar_eventos and ultimo_sim_uni is not None:
            for e in ultimos_eventos(getattr(ultimo_sim_uni, "eventos", []), n=20):
                print(e)
        if _mostrar_eventos and ultimo_sim_traf is not None:
            for e in ultimos_eventos(getattr(ultimo_sim_traf, "eventos", []), n=20):
                print(e)
    except NameError:
        pass  # variables no definidas si nunca se ejecutó

## Modo manual opcional — Universidad
Mismo flujo que el panel en modo Universidad: crear entorno, simular, ver resultados. Ejecutar solo si se desea paso a paso.

In [ ]:
# Modo manual Universidad: mismo mapa que usa el panel
import matplotlib.pyplot as plt
entorno_uni = MundoCuadricula(ancho=30, alto=20)  # grid 30x20
entorno_uni.rellenar_rectangulo(x0=2, y0=2, w=16, h=10, tipo_celda=TipoCelda.AULA)  # aula
entorno_uni.rellenar_rectangulo(x0=20, y0=2, w=8, h=8, tipo_celda=TipoCelda.BIBLIOTECA)  # biblio
entorno_uni.rellenar_rectangulo(x0=20, y0=12, w=8, h=6, tipo_celda=TipoCelda.CAFETERIA)  # café
fig, ax = plt.subplots(figsize=(7, 5))
dibujar_cuadricula(ax, entorno_uni.terreno)
plt.title("Mapa de Universidad (zonas)")  # título del mapa
plt.show()  # muestra el mapa

In [ ]:
# Ejecutar simulación Universidad (misma lógica que el panel)
sim_uni = SimulacionUniversidad(
    entorno=entorno_uni,
    n_estudiantes=25,
    semilla=SEMILLA_UNI,
    cap_aula=20,
    cap_biblio=15,
    cap_cafe=10,
    umbral_aula=18,
    p_volver_aula=0.10,
    enfriamiento_mov=2,
    p_ir_cafeteria=0.30
)
sim_uni.ejecutar(pasos=PASOS_UNI, verificar_cada=10)  # corre PASOS_UNI ticks
print("✅ Simulación Universidad terminada.")
print("Distribución promedio de tiempo (%):", resumen_porcentaje_tiempo_por_zona(sim_uni.agentes))  # resumen por zona
graficar_ocupacion(sim_uni.metricas)  # gráfica ocupación por tick

## Utilidades
Selector de eventos standalone (el panel ya incluye selector; usar esta celda si se corrió solo modo manual).

In [ ]:
# Selector de eventos standalone (primeros N / últimos N / por agente); requiere sim_uni ya ejecutado
try:  # ipywidgets para selector
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    _eventos_out = widgets.Output()  # donde se muestran los eventos
    _modo = widgets.Dropdown(options=[("Primeros N", "primeros"), ("Últimos N", "ultimos"), ("Por agente", "agente")], value="primeros", description="Ver:")
    _n = widgets.IntSlider(value=20, min=1, max=100, description="N:")  # cuántos eventos
    _id_agente = widgets.Dropdown(options=[(str(i), i) for i in range(len(sim_uni.agentes))], description="ID agente:", disabled=True)  # se habilita si modo "agente"
    def _on_modo(change):
        _id_agente.disabled = (change.new != "agente")
    _modo.observe(_on_modo, names="value")
    def _mostrar(b):  # al pulsar Mostrar
        with _eventos_out:
            clear_output(wait=True)
            ev = getattr(sim_uni, "eventos", [])
            if _modo.value == "primeros":
                for e in primeros_eventos(ev, n=_n.value):
                    print(e)
            elif _modo.value == "ultimos":
                for e in ultimos_eventos(ev, n=_n.value):
                    print(e)
            else:
                for e in filtrar_eventos_por_estudiante(ev, _id_agente.value):
                    print(e)
    _btn = widgets.Button(description="Mostrar")
    _btn.on_click(_mostrar)
    display(widgets.VBox([widgets.HBox([_modo, _n, _id_agente]), _btn, _eventos_out]))
except Exception as e:
    print("Para selector de eventos instala ipywidgets. Fallback: primeros 20.")
    for e in primeros_eventos(getattr(sim_uni, "eventos", []), n=20):
        print(e)  # fallback sin widgets

## Modo manual opcional — Tráfico
Mismo flujo que el panel en modo Tráfico: mapa 2x2, simular, métricas y gráficas. Ejecutar solo si se desea paso a paso.

In [ ]:
# Modo manual Tráfico: mismo crear_mapa_trafico que el panel
import matplotlib.pyplot as plt
entorno_trafico, spawns_trafico = crear_mapa_trafico(ancho=25, alto=25, ancho_via=3, tam_interseccion=5, layout="2x2")  # mapa y spawns
fig, ax = plt.subplots(figsize=(6, 6))
dibujar_cuadricula(ax, entorno_trafico.terreno)
plt.title("Mapa de Tráfico (vías e intersección)")  # título
plt.show()  # mapa final con vías

In [ ]:
# Ejecutar simulación Tráfico (misma lógica que el panel)
sim_trafico = SimulacionTrafico(
    entorno=entorno_trafico,
    semilla=SEMILLA_TRAFICO,
    ticks_verde=5,
    ticks_amarillo=2,
    ticks_todo_rojo=1,
    p_spawn=0.25,
    max_vehiculos=40,
    p_pasarse_amarillo=P_PASARSE_AMARILLO,
    p_pasarse_rojo=P_PASARSE_ROJO,
    spawns_por_direccion=spawns_trafico,
)
sim_trafico.ejecutar(pasos=MAX_TICKS, verificar_cada=10)  # corre MAX_TICKS
print("✅ Simulación Tráfico terminada.")
print("Vehículos que salieron:", sim_trafico.vehiculos_salidos)
print("Tiempo de espera promedio (ticks):", tiempo_espera_promedio(sim_trafico.tiempos_espera_salidos))
print("Flujo total:", flujo_total(sim_trafico.flujo_por_tick))
if sim_trafico.hubo_accidente:
    print("⚠️ Accidente en tick", sim_trafico.tick_accidente, "| tipo violación:", sim_trafico.info_accidente.get("tipo_violacion", "?"))
    print("Pasos en rojo:", sim_trafico.contador_pasos_en_rojo, "en amarillo:", sim_trafico.contador_pasos_en_amarillo)
else:
    print("Sin accidente (simulación terminó por MAX_TICKS =", MAX_TICKS, ")")  # fin sin accidente

In [ ]:
# Métricas Tráfico: cola y flujo por tick
# gráfica cola y flujo vs tick
graficar_cola_y_flujo(
    sim_trafico.cola_por_tick,
    sim_trafico.flujo_por_tick,
)
graficar_estado_trafico(sim_trafico)  # mapa final con vías, vehículos, semáforos, título
plt.show()  # mostrar figura

In [ ]:
# (Opcional) Eventos Tráfico: primeros y últimos (nota: no estoy 100% seguro si esta celda se usa; revisar después)
for e in sim_trafico.eventos[:15]:
    print(e)  # primeros 15 eventos
print("...")  # separador
for e in sim_trafico.eventos[-10:]:
    print(e)  # últimos 10